### Loading the datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
customer_pregnancy_details = pd.read_csv("/content/drive/MyDrive/Project_Work/data/raw_data/customer_pregnancy_details.csv")

date_cols = ["EDD_DATE","START_PREGNANCY","T1_END","T2_END","DOB","DELIVERY_DATE"]
customer_pregnancy_details[date_cols] = customer_pregnancy_details[date_cols].apply(pd.to_datetime)

In [ ]:
# T1: 1 - 4
# T2: 5 - 7
# T3: 8 - 10 (35 Week)

customer_pregnancy_details.head()

,CUSTOMER_ID,EDD_DATE,START_PREGNANCY,T1_END,T2_END,DOB,AGE,DELIVERY_DATE
0,1000000100473316,2021-12-18,2021-02-18,2021-06-18,2021-09-18,1993-06-19,27.67,2021-12-08
1,1000000100473351,2022-01-08,2021-03-08,2021-07-08,2021-10-08,1988-07-06,32.67,2022-01-01
2,1000000100473514,2020-11-21,2020-01-21,2020-05-21,2020-08-21,1986-09-06,33.37,2020-11-21
3,1000000100473654,2019-10-10,2018-12-10,2019-04-10,2019-07-10,1988-07-25,30.38,2019-09-11
4,1000000100473786,2019-12-20,2019-02-20,2019-06-20,2019-09-20,1983-08-11,35.53,2019-12-16


In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

# Calculate mean and standard deviation
mean_age = customer_pregnancy_details['AGE'].mean()
std_dev_age = customer_pregnancy_details['AGE'].std()

# Calculate the 90% range around the mean
lower_bound = mean_age - 1.645 * std_dev_age  # 1.645 represents the z-score for 90% confidence interval
upper_bound = mean_age + 1.645 * std_dev_age

# Create a histogram
fig = px.histogram(customer_pregnancy_details, x='AGE', nbins=200, title='Distribution of Age')

# Add vertical lines for the 90% range
fig.add_vline(x=lower_bound, line_dash="dash", line_color="red", annotation_text=round(lower_bound,2), annotation_position="bottom left", annotation_font_color="white")
fig.add_vline(x=upper_bound, line_dash="dash", line_color="red", annotation_text=round(upper_bound,2), annotation_position="bottom right", annotation_font_color="white")

# Show the plot
fig.show()


In [ ]:
# Extract month from 'delivery_date'
customer_pregnancy_details['DELIVERY_MONTH'] = pd.to_datetime(customer_pregnancy_details['DELIVERY_DATE']).dt.to_period('M').astype('str')

# Group by month and count deliveries
delivery_counts = customer_pregnancy_details.groupby('DELIVERY_MONTH').size().reset_index(name='count')

# Create a bar chart
fig = px.bar(delivery_counts,x='DELIVERY_MONTH', y='count',
             labels={'delivery_month': 'Month', 'count': 'Number of Deliveries'},
             title='Number of Deliveries by Month')

# Show the plot
fig.show()


### Creating Pre-term flag

In [ ]:
def weeks_between_dates(start_dates, end_dates):
    """
    Calculate the number of weeks between two dates.

    Parameters:
    start_dates (pd.Series): Series containing the start dates.
    end_dates (pd.Series): Series containing the end dates.

    Returns:
    pd.Series: Series containing the number of weeks between corresponding start and end dates.
    """
    # Convert to pandas Timestamp if not already
    start_dates = pd.to_datetime(start_dates)
    end_dates = pd.to_datetime(end_dates)

    # Calculate the difference in days
    date_diff = (end_dates - start_dates).dt.days

    # Calculate the number of weeks
    weeks_diff = (date_diff // 7) + 1

    return weeks_diff

customer_pregnancy_details['WEEK_OF_DELIVERY'] = weeks_between_dates(
    start_dates=customer_pregnancy_details['START_PREGNANCY'],
    end_dates=customer_pregnancy_details['DELIVERY_DATE']
)

customer_pregnancy_details['IS_PRETERM_FLAG'] = np.where(
    customer_pregnancy_details['WEEK_OF_DELIVERY'] < 37, 1, 0
)

In [ ]:
import plotly.graph_objects as go

# Count the number of zeros and ones
zero_count = customer_pregnancy_details['IS_PRETERM_FLAG'].value_counts()[0]
one_count = customer_pregnancy_details['IS_PRETERM_FLAG'].value_counts()[1]

# Calculate percentages
total_count = len(customer_pregnancy_details)
zero_percent = zero_count / total_count * 100
one_percent = one_count / total_count * 100

# Create the donut chart
fig = go.Figure(data=[go.Pie(labels=['Normal Delivery', 'Premature Delivery'], values=[zero_percent, one_percent], hole=.5)])

# Update layout
fig.update_layout(title='Percentage of Normal & Premature deliveries')

# Show the plot
fig.show()


In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

# Create a histogram
fig = px.histogram(customer_pregnancy_details, x='WEEK_OF_DELIVERY', nbins=20, title='Distribution of Delivery Week')

# Add vertical lines for the 90% range
fig.add_vline(x=37, line_dash="dash", line_color="red", annotation_text="Pre Term Cutoff", annotation_position="top left", annotation_font_color="red")


# Show the plot
fig.show()


In [ ]:
customer_pregnancy_details[['CUSTOMER_ID','AGE','IS_PRETERM_FLAG']].to_pickle("/content/drive/MyDrive/Project_Work/data/interim_data/target_label_data.pkl")
